# PlannerAgent GCC-4K — Student v0.2
Select a GPU runtime, upload the prepared GCC4K INPUT ZIP, and Run all.

In [ ]:
import os, platform, shutil, subprocess, sys
subprocess.run(['nvidia-smi'],check=True)
import torch
assert torch.cuda.is_available(), 'CUDA GPU required'
print(platform.platform(),torch.cuda.get_device_name(0),torch.version.cuda,sys.version,shutil.disk_usage('/content'))

In [ ]:
import importlib.util, subprocess, sys
if importlib.util.find_spec('torchao') is not None:
    subprocess.run([sys.executable,'-m','pip','uninstall','-y','torchao'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==4.51.3','peft==0.20.0','accelerate','safetensors','psutil'],check=True)
os.environ['WANDB_DISABLED']='true'; os.environ['HF_HUB_DISABLE_TELEMETRY']='1'

In [ ]:
from google.colab import files
uploaded=files.upload(); archives=[name for name in uploaded if name.endswith('.zip')]
assert len(archives)==1,'Upload exactly one GCC4K INPUT ZIP'
shutil.unpack_archive(archives[0],'/content/gcc4k')

In [ ]:
import hashlib, pathlib
root=pathlib.Path('/content/gcc4k')
for line in (root/'SHA256SUMS.txt').read_text().splitlines():
    expected,relative=line.split('  ',1); assert '\\' not in relative; assert hashlib.sha256((root/relative).read_bytes()).hexdigest()==expected,relative
print('Bundle integrity and POSIX paths: PASS')

In [ ]:
!cd /content/gcc4k/scripts && python train_targeted_student_v02.py

In [ ]:
result='/content/gcc4k/PA-INTERPRETATION-STUDENT-v0.2-GCC4K.zip'
assert os.path.exists(result)
print(result,os.path.getsize(result),hashlib.sha256(open(result,'rb').read()).hexdigest())
files.download(result)